In [93]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import layers,Model
from tensorflow.keras.applications import MobileNetV2,ResNet50V2,EfficientNetB0
data,info=tfds.load('plant_village',split='train',with_info=True,as_supervised=True)

In [94]:
data_size=info.splits["train"].num_examples
class_names=info.features["label"].names
n_classes=info.features["label"].num_classes
print("data size :",data_size,"\nclass_names:",class_names,"\n classes :",n_classes)
print("GPU: ",len(tf.config.list_physical_devices("GPU")))

data size : 54303 
class_names: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry___healthy', 'Cherry___Powdery_mildew', 'Corn___Cercospora_leaf_spot Gray_leaf_spot', 'Corn___Common_rust', 'Corn___healthy', 'Corn___Northern_Leaf_Blight', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___healthy', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___healthy', 'Potato___Late_blight', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___healthy', 'Strawberry___Leaf_scorch', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___healthy', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___

In [95]:
for image, label in data.take(1):
    print('Image shape:', image.shape)
    print('Label:', label.numpy())

Image shape: (256, 256, 3)
Label: 35


In [96]:
train_data=data.take(round(0.8*data_size))
test_data=data.skip(round(0.8*data_size))

In [97]:
AUTOTUNE = tf.data.AUTOTUNE
def get_dataset(model_name,train_data,test_data):
    models={'MobileNetV2': tf.keras.applications.mobilenet_v2.preprocess_input,
        'ResNet50':tf.keras.applications.resnet50.preprocess_input,
        'EfficientNetB0':tf.keras.applications.efficientnet.preprocess_input}
    preprocess_input=models[model_name]
    def preprocess(image,label):
        image=tf.image.resize(image,(160,160))
        image=tf.cast(image,tf.float32) 
        image = preprocess_input(image)
        label=tf.one_hot(label,depth=38)
        return image,label
    augumentation=tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(.2),
        layers.RandomZoom(.2)
    ])
    train_p_data=train_data.map(preprocess, tf.data.AUTOTUNE).map(lambda image,label:(augumentation(image,training=True),label),
                                num_parallel_calls=AUTOTUNE).shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)
    test_p_data=test_data.map(preprocess, tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)
    return train_p_data,test_p_data

In [98]:
for image, label in train_p_data.take(1):
    print("Image min:", tf.reduce_min(image).numpy())
    print("Image max:", tf.reduce_max(image).numpy())

Image min: -1.0
Image max: 0.999842


**MobileNetV2**

In [99]:
def build_model(model_backbone,train_base=False):
    backbone={'MobileNetV2': tf.keras.applications.MobileNetV2,       
        'ResNet50': tf.keras.applications.ResNet50,               
        'EfficientNetB0': tf.keras.applications.EfficientNetB0}
    base=backbone[model_backbone](input_shape=(160,160,3),include_top=False,weights="imagenet")

    base.trainable=train_base
    inputs=tf.keras.Input(shape=(160,160,3))
    x=base(inputs,training=False)
    x=layers.GlobalAveragePooling2D()(x)
    x=layers.Dense(256,activation='relu')(x)
    x=layers.Dropout(.2)(x)
    outputs=layers.Dense(38,activation='softmax')(x)
    return Model(inputs,outputs),base
train_p_data,test_p_data=get_dataset('MobileNetV2',train_data,test_data)
model,base=build_model('MobileNetV2')
model.compile(optimizer=tf.keras.optimizers.Adam(.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary() 

Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_22 (InputLayer)     │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_5      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 38)             │         9,766 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,595,686 (9.90 MB)

 Trainable params: 337,702 (1.29 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:
callback=EarlyStopping(
    monitor='val_loss',   
    patience=2,           
    restore_best_weights=True  
)
history1 = model.fit(
    train_p_data,
    validation_data=test_p_data,
    epochs=10,callbacks=callback)

Epoch 1/10


I0000 00:00:1782733201.506609   21626 service.cc:153] XLA service 0x7fc50805df10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782733201.506669   21626 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2050, Compute Capability 8.6 (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.23.2)
I0000 00:00:1782733201.672271   21626 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1782733202.700956   21626 cuda_dnn.cc:461] Loaded cuDNN version 92302
I0000 00:00:1782733202.815426   21626 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8511__.123
I0000 00:00:1782733205.190701   21626 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0

   2/1358 ━━━━━━━━━━━━━━━━━━━━ 1:44 77ms/step - accuracy: 0.0078 - loss: 4.3860       

I0000 00:00:1782733222.651533   21626 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1356/1358 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.5734 - loss: 1.6792

...

1358/1358 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.5736 - loss: 1.6782

I0000 00:00:1782733354.209344   21625 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.


1358/1358 ━━━━━━━━━━━━━━━━━━━━ 185s 116ms/step - accuracy: 0.7279 - loss: 1.0228 - val_accuracy: 0.8728 - val_loss: 0.4448
Epoch 2/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 137s 98ms/step - accuracy: 0.8745 - loss: 0.4127 - val_accuracy: 0.8995 - val_loss: 0.3334
Epoch 3/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 124s 90ms/step - accuracy: 0.9013 - loss: 0.3213 - val_accuracy: 0.9081 - val_loss: 0.2904
Epoch 4/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 119s 87ms/step - accuracy: 0.9142 - loss: 0.2685 - val_accuracy: 0.9154 - val_loss: 0.2583
Epoch 5/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 117s 85ms/step - accuracy: 0.9232 - loss: 0.2418 - val_accuracy: 0.9147 - val_loss: 0.2606
Epoch 6/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 115s 83ms/step - accuracy: 0.9308 - loss: 0.2179 - val_accuracy: 0.9239 - val_loss: 0.2320
Epoch 7/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 116s 84ms/step - accuracy: 0.9329 - loss: 0.2071 - val_accuracy: 0.9255 - val_loss: 0.2258
Epoch 8/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 118s 86ms/step - accuracy: 0.9365 - l

W0000 00:00:1782734473.770011   23787 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 16782080 bytes after encountering the first element of size 16782080 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


1358/1358 ━━━━━━━━━━━━━━━━━━━━ 132s 96ms/step - accuracy: 0.9439 - loss: 0.1747 - val_accuracy: 0.9329 - val_loss: 0.2012


In [10]:
base.trainable=True
for layer in base.layers[:100]:
    layer.trainable=False
model.compile(
    optimizer=tf.keras.optimizers.Adam(.00001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
    )
history2=model.fit(train_p_data,epochs=5,validation_data=test_p_data,callbacks=callback)

Epoch 1/5


I0000 00:00:1782734503.001997   21626 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_99102__.195


1356/1358 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.7981 - loss: 0.6766

I0000 00:00:1782734633.495412   21624 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_99102__.195


1358/1358 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.7982 - loss: 0.6762

W0000 00:00:1782734653.305651   30247 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 16782080 bytes after encountering the first element of size 16782080 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


1358/1358 ━━━━━━━━━━━━━━━━━━━━ 184s 110ms/step - accuracy: 0.8684 - loss: 0.4229 - val_accuracy: 0.9193 - val_loss: 0.2388
Epoch 2/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 165s 119ms/step - accuracy: 0.9313 - loss: 0.2115 - val_accuracy: 0.9424 - val_loss: 0.1649
Epoch 3/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 166s 120ms/step - accuracy: 0.9435 - loss: 0.1703 - val_accuracy: 0.9522 - val_loss: 0.1442
Epoch 4/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.9509 - loss: 0.1440

W0000 00:00:1782735131.525474   30247 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 16782080 bytes after encountering the first element of size 16782080 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


1358/1358 ━━━━━━━━━━━━━━━━━━━━ 143s 104ms/step - accuracy: 0.9531 - loss: 0.1398 - val_accuracy: 0.9572 - val_loss: 0.1259
Epoch 5/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 139s 101ms/step - accuracy: 0.9598 - loss: 0.1213 - val_accuracy: 0.9607 - val_loss: 0.1133


In [16]:
result={"MobileNetV2":{"train accuracy":round(max(history1.history["accuracy"]),2),
                       "train loss":round(min(history1.history["loss"]),2),
                       "validation accuracy":round(max(history2.history["accuracy"]),2),
                       "validation loss":round(min(history2.history["loss"]),2)}}
result

{'MobileNetV2': {'train accuracy': 0.94,
  'train loss': 0.17,
  'validation accuracy': 0.96,
  'validation loss': 0.12}}